In [ ]:
import pandas as pd

# Données

In [ ]:
x_df = pd.read_csv("engieX.csv", sep=";")
y_df = pd.read_csv("engieY.csv", sep=";")

Nous filtrons *WT1* uniquement.

In [ ]:
x_wt1_df = x_df[x_df.MAC_CODE == "WT1"].copy()
y_wt1_df = y_df[y_df.ID.isin(x_wt1_df.ID)].copy()

In [ ]:
x_wt1_df

In [ ]:
y_wt1_df

In [ ]:
df = x_wt1_df.merge(y_wt1_df, how="left", on="ID")
df

In [ ]:
df.drop(
    columns=[
        col 
        for col in df.columns
        if col.endswith("min") or col.endswith("max") or col.endswith("std")
    ],
    inplace=True
)

In [ ]:
df = df[df.Date_time % 6 == 1].copy()
df

In [ ]:
for col in df.columns:
    n_na = df[col].isna().sum()
    if n_na > 0:
        print(f"{col} : {n_na}")

In [ ]:
df.drop(
    columns=[
        "Generator_converter_speed",
        "Gearbox_inlet_temperature",
        "Grid_voltage",
        "Absolute_wind_direction_c",
        "Nacelle_angle_c"
    ],
    inplace=True
)

In [ ]:
# Tri par ordre chronologique
df.sort_values(by="Date_time", inplace=True)
df.reset_index(drop=True, inplace=True)

df.drop(columns=["ID", "MAC_CODE", "Date_time"], inplace=True)

## Séparation des données

In [ ]:
# On garde les 80% premières données dans le temps pour l'entraînement, le reste pour le test
cv_thres = 0.8 * len(df)
train_df = df[df.index < cv_thres].copy()
test_df = df[df.index >= cv_thres].copy()

In [ ]:
X_train = train_df[[col for col in train_df.columns if col != "TARGET"]]
y_train = train_df[["TARGET"]]

In [ ]:
X_test = test_df[[col for col in test_df.columns if col != "TARGET"]]
y_test = test_df[["TARGET"]]

## Modèles

In [ ]:
from sklearn.metrics import mean_absolute_error

### Régression linéaire simple

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
simple_lm_model = LinearRegression().fit(X_train, y_train)

In [ ]:
# Erreur d'entraînement
mean_absolute_error(simple_lm_model.predict(X_train), y_train)

In [ ]:
# Erreur de test
mean_absolute_error(simple_lm_model.predict(X_test), y_test)

### Forêt aléatoire

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_model = RandomForestRegressor().fit(X_train, y_train)

In [ ]:
# Erreur d'entraînement
mean_absolute_error(rf_model.predict(X_train), y_train)

In [ ]:
# Erreur de test
mean_absolute_error(rf_model.predict(X_test), y_test)

### Réseau de neurones

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
nn_model = MLPRegressor(hidden_layer_sizes=(50, 50)).fit(X_train, y_train)

In [ ]:
# Erreur d'entraînement
mean_absolute_error(nn_model.predict(X_train), y_train)

In [ ]:
# Erreur de test
mean_absolute_error(nn_model.predict(X_test), y_test)